# Exercise 03, Part 2: What is wrong with these files?

Run this in Google Colab, signed in with your WashU Google account.

Each file has missing or incorrect coordinate information. Identify the problem
and calculate the error it causes in meters. In Step 7, write and submit a
metadata record for one of the files.

Give your answers as numbers and sentences, without maps.

## Setup

Run the setup cell to install the geospatial software and download the data
bundle. Wait for it to finish before continuing.

In [ ]:
!pip -q install pyproj rasterio geopandas matplotlib
print("installed")

In [ ]:
import pathlib, urllib.request, urllib.error

REPO = "https://raw.githubusercontent.com/washu-eeps/eeps4684-ex03-data/main/data"
FILES = [
    "tisch_control.csv",
    "benchmark_coordinates.csv",
    "tisch_control_4326.gpkg",
    "benchmarks.gpkg",
    "benchmarks.shp",
    "benchmarks.shx",
    "benchmarks.dbf",
    "benchmarks.prj",
    "benchmarks.cpg",
    "tisch_2024_band5_30cm.tif"
]

pathlib.Path("data").mkdir(exist_ok=True)
missing = []
for name in FILES:
    dest = pathlib.Path("data") / name
    if dest.exists():
        continue
    try:
        urllib.request.urlretrieve(f"{REPO}/{name}", dest)
    except urllib.error.HTTPError:
        missing.append(name)

if missing:
    print("Could not download:", ", ".join(missing))
    print("Obtain eeps4684_ex03_data.zip from your instructor, unzip it, and")
    print("upload the files with the next cell instead.")
else:
    for p in sorted(pathlib.Path("data").iterdir()):
        print(f"{p.stat().st_size/1e6:8.2f} MB  {p.name}")

In [ ]:
# ONLY run this if the download above failed. Select every file in the bundle
# at once; the picker accepts a multiple selection.
import pathlib, shutil
from google.colab import files

pathlib.Path("data").mkdir(exist_ok=True)
for name in files.upload():
    shutil.move(name, pathlib.Path("data") / name)
    print("stored data/" + name)

## Step 1. Which reference frame is the control file in?

`tisch_control.csv` is the control survey the class works from. Open it and see
what it tells you.

In [ ]:
import pandas as pd

control = pd.read_csv("data/tisch_control.csv")
print(control.head().to_string(index=False))
print()
print(f"{len(control)} points")
print(f"Easting  {control.Easting.min():.1f} to {control.Easting.max():.1f}")
print(f"Northing {control.Northing.min():.1f} to {control.Northing.max():.1f}")
print(f"site spans {control.Easting.max()-control.Easting.min():.0f} m east-west")

None of the five columns records a reference frame. The file also lacks units,
an epoch, and a statement of whether `Elevation` is measured from the ellipsoid
or from sea level.

A coordinate is three numbers; a position is a location in a particular frame.
See [Metadata and provenance](https://bradleylab.github.io/geospatial-field-methods/toolchain/metadata-and-provenance).
Without a frame, this file provides coordinates but does not specify positions.

Start by ruling out one of the two projected systems used locally. Both use
meters, so the numbers are plausible in either.

In [ ]:
from pyproj import Transformer

# Two candidate frames, both NAD83-family, both in meters, both used in Missouri.
CANDIDATES = {
    "EPSG:6512":  "NAD83(2011) / Missouri East",
    "EPSG:26915": "NAD83 / UTM zone 15N",
}

for epsg, name in CANDIDATES.items():
    lon, lat = Transformer.from_crs(epsg, "EPSG:6318", always_xy=True).transform(
        control.Easting.to_numpy(), control.Northing.to_numpy())
    print(f"read as {name:28s} -> {lat.mean():8.4f} N, {lon.mean():9.4f} E")

That is check 1 from
[coordinate systems in practice](https://bradleylab.github.io/geospatial-field-methods/foundations/crs-in-practice):
*is it on the right continent?* One candidate places the survey on the Danforth
campus; the other places it several hundred kilometers out in the Pacific.

Ruling out one candidate does not establish the other. That requires an
independent measurement of the same ground. `benchmark_coordinates.csv` contains
four monumented benchmarks with surveyed, published coordinates independent of
this course.

In [ ]:
bench = pd.read_csv("data/benchmark_coordinates.csv")
print(list(bench.columns))
print()
print(bench[["OID_", "Easting", "Northing", "NAD83_2011_X", "NAD83_2011_Y",
             "EPSG6318_X", "EPSG6318_Y", "ElvGeo03A"]].to_string(index=False))

The benchmark file gives coordinates for the same four points in several frames
but does not identify the frame of each column. If two of these benchmarks also
appear in the control file, the two files describe the same physical marks, so
comparing their coordinates tests a proposed frame for the control file.

In [ ]:
import numpy as np

# Convert the control points to NAD83(2011) latitude and longitude, ASSUMING
# the frame you did not rule out, then look for benchmarks nearby.
lon, lat = Transformer.from_crs("EPSG:6512", "EPSG:6318", always_xy=True).transform(
    control.Easting.to_numpy(), control.Northing.to_numpy())

for _, b in bench.iterrows():
    m_per_deg_e = 111_320 * np.cos(np.radians(b.EPSG6318_Y))
    d = np.hypot((lon - b.EPSG6318_X) * m_per_deg_e,
                 (lat - b.EPSG6318_Y) * 111_320)
    i = int(d.argmin())
    near = f"{d[i]*1000:6.1f} mm" if d[i] < 1 else f"{d[i]:6.1f} m "
    print(f"benchmark {int(b.OID_)}: nearest control point {control.Name[i]:5s} "
          f"at {near}   ({control.Description[i]})")

All four benchmarks were matched against the nearest control point. Decide how
many of the four are genuine matches and which are not, and say what separates
them.

Record the genuine matches with their closures in millimeters, and give the EPSG
code they support. A nearest-neighbor search returns a result for every input,
so state what you would have concluded from the distances alone if you had not
checked their size.

## Step 2. What are the units, and what error does using the wrong unit cause?

Compare the benchmark file's `Easting`/`Northing` pair with its
`NAD83_2011_X`/`_Y` pair. They describe the same four points in the same
projection but use different units.

Determine the unit from the coordinates.

In [ ]:
ratio_e = (bench.Easting / bench.NAD83_2011_X).to_numpy()
ratio_n = (bench.Northing / bench.NAD83_2011_Y).to_numpy()

print(f"easting  ratio {ratio_e.mean():.7f}   spread {ratio_e.std():.1e}")
print(f"northing ratio {ratio_n.mean():.7f}   spread {ratio_n.std():.1e}")
print()
print(f"US survey foot  1200/3937 m  ->  1 m = {3937/1200:.7f} ft")
print(f"international foot 0.3048 m ->  1 m = {1/0.3048:.7f} ft")

The two candidate feet differ in the sixth decimal place, or two parts per
million. The resulting error depends on the coordinate's magnitude, and a State
Plane easting has six figures.

Calculate how far the point moves when the file's feet are interpreted as the
wrong foot.

In [ ]:
INTL_FT = 0.3048              # exact, by definition
US_FT = 1200 / 3937           # exact, by definition

correct = bench.Easting * US_FT
wrong = bench.Easting * INTL_FT

for _, row in pd.DataFrame({"oid": bench.OID_, "ft": bench.Easting,
                            "correct_m": correct, "wrong_m": wrong}).iterrows():
    print(f"benchmark {int(row.oid)}: {row.ft:12.3f} ft -> "
          f"{row.correct_m:11.3f} m correct, {row.wrong_m:11.3f} m wrong, "
          f"off by {row.correct_m - row.wrong_m:+.3f} m")

Record the error and compare it with the RTK precision you measured in
Exercise 01.

A datum problem and a units problem both appear as a systematic offset, and over
a site 250 m across both look like a constant shift. Given only a set of shifted
coordinates, what test distinguishes them? Compare how the offset changes
between the four benchmarks.

## Step 3. How far apart are NAD83 and WGS84 at this site?

In Exercise 01 you converted a position to WGS84 and reported the tool's stated
transformation accuracy. Measure the shift itself using the benchmark file,
which gives coordinates for the same four marks in both frames.

`EPSG6318_X`/`_Y` are NAD83(2011). `WGS84_LON_dd`/`_LAT_dd` are the same four
marks in WGS84. Subtract them.

In [ ]:
lat0 = bench.EPSG6318_Y.to_numpy()
m_per_deg_e = 111_320 * np.cos(np.radians(lat0))

de = (bench.WGS84_LON_dd.to_numpy() - bench.EPSG6318_X.to_numpy()) * m_per_deg_e
dn = (bench.WGS84_LAT_dd.to_numpy() - bench.EPSG6318_Y.to_numpy()) * 111_320

print("WGS84 minus NAD83(2011), from the file's own columns:")
for oid, e, n in zip(bench.OID_, de, dn):
    print(f"  benchmark {int(oid)}: east {e:+.3f} m  north {n:+.3f} m  "
          f"total {np.hypot(e, n):.3f} m")

Report the shift magnitude and direction at each mark. How consistent are they?

For check 4 from the reference text, request the same transformation from PROJ
and compare the results.

In [ ]:
# WGS84 is a family of realizations, not one frame. Ask for a specific one.
to_g1762 = Transformer.from_crs("EPSG:6318", "EPSG:9057", always_xy=True)
lo2, la2 = to_g1762.transform(bench.EPSG6318_X.to_numpy(), lat0)

pe = (lo2 - bench.EPSG6318_X.to_numpy()) * m_per_deg_e
pn = (la2 - lat0) * 111_320

print(f"PROJ, NAD83(2011) -> WGS84 (G1762)")
print(f"   applies      east {pe.mean():+.3f} m  north {pn.mean():+.3f} m  "
      f"total {np.hypot(pe, pn).mean():.3f} m")
print(f"   claims accuracy {to_g1762.accuracy} m")
print(f"   disagrees with the file by {np.hypot(de-pe, dn-pn).mean()*1000:.1f} mm")

Report the disagreement between the two routes in millimeters. How closely do
they agree on your installation?

Request the transformation to plain "WGS84" without naming a realization.

In [ ]:
for dst, label in [("EPSG:9057", 'WGS84 (G1762), a named realization'),
                   ("EPSG:4326", 'WGS 84, the generic code')]:
    t = Transformer.from_crs("EPSG:6318", dst, always_xy=True)
    lo2, la2 = t.transform(bench.EPSG6318_X.to_numpy(), lat0)
    shift = np.hypot((lo2 - bench.EPSG6318_X.to_numpy()) * m_per_deg_e,
                     (la2 - lat0) * 111_320).mean()
    print(f"{label:36s} applies {shift:.3f} m, claims accuracy {t.accuracy} m")
    print(f"{'':36s} operation: {t.description[:80]}")
    print()

Report the shift and stated accuracy for each destination CRS on your installation.
Do the coordinates change in both cases? Distinguish the size of an applied shift
from the stated accuracy of the operation. If accuracy is reported as -1, record
it as unknown, not as a negative distance.

Generic `EPSG:4326` does not specify a particular WGS84 realization. Inspect the
operation printed above: does it apply a shift or use an approximation that
leaves the coordinates unchanged? Software versions and installed transformation
resources can affect the result; report what you observe.

Which destination would you use, and what information about realization and epoch
would you need to justify it? What can you infer from a published coordinate
labeled only "WGS84", and what remains unknown?

## Step 4. A file that states its frame, incorrectly

`tisch_control_4326.gpkg` contains the control survey converted to latitude and
longitude. Unlike the CSVs, a GeoPackage has a field for the reference frame.
This file's field says WGS 84, but its coordinates are NAD83(2011).

The `.pos` file Emlid Studio wrote in Exercise 02 also labels its coordinates
WGS84. A PPK solution uses the frame of its base station, which for the MoDOT
network is NAD83(2011) at epoch 2010.00. The `.pos` file does not record this
distinction.

In [ ]:
import geopandas as gpd

g = gpd.read_file("data/tisch_control_4326.gpkg")
print(f"declared CRS : {g.crs.to_string()}  ({g.crs.name})")
print(f"features     : {len(g)}")
print(g.head(3)[["Name", "geometry"]].to_string(index=False))

Project the coordinates back to the frame established in Step 1 twice: once
using the labeled frame and once using the actual frame.

In [ ]:
lon_g, lat_g = g.geometry.x.to_numpy(), g.geometry.y.to_numpy()

believe = Transformer.from_crs("EPSG:4326", "EPSG:6512", always_xy=True)
truth = Transformer.from_crs("EPSG:6318", "EPSG:6512", always_xy=True)

be, bn = believe.transform(lon_g, lat_g)
te, tn = truth.transform(lon_g, lat_g)

print(f"believing the label : accuracy {believe.accuracy} m")
print(f"knowing the truth   : accuracy {truth.accuracy} m")
print(f"coordinates differ by {np.hypot(be-te, bn-tn).max()*1000:.3f} mm")
print()
print(f"and the correct route reproduces tisch_control.csv to "
      f"{np.hypot(te-control.Easting.to_numpy(), tn-control.Northing.to_numpy()).max()*1000:.2f} mm")

Compare the two routes on both counts: the coordinates they produce and the
accuracy each one claims. State which of the two changed and which did not.

Then say in one sentence what the mislabel cost, and consider a dataset that has
passed through several hands: if the label is wrong and the numbers never move,
at what point in the chain could anyone have caught it?

## Step 5. Undeclared nodata

Now a raster. `tisch_2024_band5_30cm.tif` is one band of the five-band
orthophoto flown over campus by a previous class in October 2024, resampled to
30 cm so it fits in this notebook.

The flight covered an irregular polygon, but a raster is a rectangle, so the
corners outside the flight have to hold something.

In [ ]:
import rasterio

with rasterio.open("data/tisch_2024_band5_30cm.tif") as src:
    band = src.read(1)
    print(f"size          {src.width} x {src.height}")
    print(f"dtype         {src.dtypes[0]}")
    print(f"CRS           {src.crs}")
    print(f"declared nodata {src.nodata}")
    print(f"band name     {src.descriptions[0]}")

The file declares no nodata value, so the unmasked mean below counts every cell.
The most common value is only a candidate fill value: frequency alone does not
establish nodata. Inspect its spatial pattern before excluding it.

Supplied provenance for this exercise: the source orthophoto has zero fill outside
the flight footprint, but leaves nodata undeclared. For this exercise, treat zero
as fill. Record that interpretation explicitly; do not generalize the rule to
other rasters or infer a fill value from frequency alone.

In [ ]:
values, counts = np.unique(band, return_counts=True)
candidate_fill = values[counts.argmax()]

import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.imshow(band == candidate_fill, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
ax.set(title="Candidate fill locations (white)", xlabel="Column (pixels)", ylabel="Row (pixels)")
plt.show()
print(f"Candidate fill value: {candidate_fill}")
print("Inspect whether the candidate occupies the exterior of the flight footprint.")
print("Supplied provenance identifies zero as fill for this exercise.")

Compare the candidate mask with the supplied provenance. Explain why the mask
supports (or conflicts with) the fill interpretation. The next cell explicitly
excludes zero using that provenance; it does not automatically exclude the mode.

In [ ]:
fill = 0  # supplied exercise provenance; do not infer this from frequency alone
fill_count = np.count_nonzero(band == fill)

print(f"Provenance-based fill value: {fill}, excluded in {fill_count:,} cells "
      f"({100*fill_count/band.size:.1f}% of the grid)")
print()
print(f"mean, fill counted as data : {band.mean():10.1f}")
print(f"mean, fill excluded        : {band[band != fill].mean():10.1f}")
print(f"the undeclared fill biases the mean "
      f"{100*(band[band != fill].mean()-band.mean())/band[band != fill].mean():.1f}% low")

Report the fraction of the grid that is not data, the band's mean with the fill
excluded, and the one line in the file's header that would have prevented the
error.

This is band 5 of 5, and its name in the file is `Band_5`. Nothing in the file
records what wavelength it holds. In Session 9 you will compute vegetation
indices, which are ratios of specific bands, from a file like this one. What
would you have to find out first, where would you look, and what happens to the
index if you guess wrong?

## Step 6. What a shapefile drops

Two files, `benchmarks.gpkg` and `benchmarks.shp`, hold the same four benchmarks
written out of the same table by the same command. Compare their columns.

In [ ]:
gpkg = gpd.read_file("data/benchmarks.gpkg")
shp = gpd.read_file("data/benchmarks.shp")

print(f"{'GeoPackage':22s}  {'shapefile':22s}")
for a, b in zip(gpkg.columns, shp.columns):
    flag = "   <-- renamed" if a != b else ""
    print(f"{a:22s}  {b:22s}{flag}")
print()
print(f"are the values intact? "
      f"{np.allclose(gpkg.NAD83_2011_X, shp.NAD83_2011)}")

The shapefile format limits attribute names to ten characters. Longer names were
shortened, with a digit appended where needed to keep them distinct. All values
were retained.

Given only the shapefile, which of the two renamed coordinate columns is the
easting and which is the northing? Say how you would decide, and whether your
method would still work at a site where easting and northing are similar
numbers.

Count how many files each format wrote in `data/`.

## Step 7. Write the metadata record

Write and submit a metadata record for `tisch_control.csv`. Use the
[minimum record table](https://bradleylab.github.io/geospatial-field-methods/toolchain/metadata-and-provenance)
in the reference text. Fill in every line you can establish from the work above
or from the Exercise 01 materials. For each remaining line, state what is
unknown and what you would have to ask, rather than leaving it blank. Do not
enter a guess.

You established the horizontal frame, but the work here does not establish the
reference for `Elevation`. There is a specific reason the benchmark file cannot
resolve this.

In [ ]:
# Write your record here, then run the cell to save it. This is a deliverable:
# download it and attach it to your report.
record = '''
tisch_control.csv
=================

What was measured :
Instrument        :
Configuration     :
When              :
Where             :
Horizontal datum  :
Projection        :
Units             :
Epoch             :
Vertical datum    :
Geoid model       :
Corrections       :
Claimed accuracy  :
Observer          :
Processing so far  :

Established how:

Still unknown, and who to ask:
'''

with open("tisch_control.csv.txt", "w") as f:
    f.write(record.strip() + "\n")
print(record)

In [ ]:
from google.colab import files
files.download("tisch_control.csv.txt")

## What you should have by the end

Include answers to every question in Steps 1–7 in your PDF, not only the
summary results. The Part 2 summary should include:

- The EPSG code of the control file and the closure in millimeters that supports it.
- The units of the benchmark file and the error from using the wrong foot, in meters.
- The NAD83-to-WGS84 shift at this site, measured twice.
- The two stated accuracies from Step 4.
- The corrected mean of the raster band.

Download `tisch_control.csv.txt` and submit it as a separate text file. Also
include the fill fraction and interpretation, the band-identity discussion, and
the Step 6 format comparison in your supporting answers. Follow the handout’s
complete submission checklist for Parts 1 and 3.

In Part 3, use the distinction between coordinates and stated accuracy from
Step 4 to interpret the georeferencing residuals.